<a href="https://colab.research.google.com/github/paulantara73/ev_charge/blob/main/Snippets_Importing_libraries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/psf/requests.git

fatal: destination path 'requests' already exists and is not an empty directory.


In [4]:
!ls /content/requests

AUTHORS.rst  HISTORY.md  MANIFEST.in	 README.md	       src
docs	     LICENSE	 NOTICE		 requirements-dev.txt  tests
ext	     Makefile	 pyproject.toml  setup.py	       tox.ini


In [5]:
REPO_PATH = "/content/requests"

import os
print(os.path.exists(REPO_PATH))

True


In [6]:
!pip install networkx pandas numpy matplotlib

In [8]:
import ast
import os
import networkx as nx

REPO_PATH = "/content/requests"

G = nx.DiGraph()

for root, dirs, files in os.walk(REPO_PATH):

    # Ignore unnecessary folders
    dirs[:] = [d for d in dirs if d not in
               [".git", "__pycache__", "tests"]]

    for file in files:

        if file.endswith(".py"):

            file_path = os.path.join(root, file)

            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    source = f.read()

                tree = ast.parse(source)

                file_node = file_path.replace(REPO_PATH, "")
                G.add_node(file_node, type="File")

                for node in ast.walk(tree):

                    if isinstance(node, ast.FunctionDef):
                        function_name = node.name
                        function_id = file_node + "::" + function_name

                        G.add_node(function_id, type="Function")
                        G.add_edge(file_node, function_id,
                                   relation="DEFINED_IN")

                    elif isinstance(node, ast.ClassDef):
                        class_name = node.name
                        class_id = file_node + "::" + class_name

                        G.add_node(class_id, type="Class")
                        G.add_edge(file_node, class_id,
                                   relation="DEFINED_IN")

            except Exception as e:
                print("Skipped:", file_path, e)

print("Graph construction completed!")
print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

Graph construction completed!
Total nodes: 303
Total edges: 281


In [9]:
import pickle

with open("/content/code_graph.pkl", "wb") as f:
    pickle.dump(G, f)

print("Graph saved successfully!")

Graph saved successfully!


In [10]:
from collections import Counter

node_types = Counter(
    data["type"] for _, data in G.nodes(data=True)
)

print(node_types)

Counter({'Function': 228, 'Class': 53, 'File': 22})


In [12]:
missing_nodes = [
    node for node, data in G.nodes(data=True)
    if "type" not in data
]

print("Nodes without a type:", len(missing_nodes))
print(missing_nodes[:20])

Nodes without a type: 75
['sys', 'setuptools', 'os', 'requests', 'pygments.style', 'pygments.token', 'json', 'platform', 'ssl', 'typing', 'idna', 'urllib3', 'charset_normalizer', 'chardet', 'urllib3.contrib', 'cryptography', 'OpenSSL', '__future__', 'urllib3.exceptions', 'compat']


In [13]:
for node in missing_nodes:
    G.nodes[node]["type"] = "Module"

print("Missing node types fixed!")

Missing node types fixed!


In [14]:
from collections import Counter

print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

print("\nNode types:")
print(Counter(data["type"] for _, data in G.nodes(data=True)))

print("\nRelationship types:")
print(Counter(data["relation"] for _, _, data in G.edges(data=True)))

Total nodes: 385
Total edges: 963

Node types:
Counter({'Function': 241, 'Module': 75, 'Class': 47, 'File': 22})

Relationship types:
Counter({'CALLS': 472, 'DEFINED_IN': 288, 'IMPORTS': 203})


In [15]:
import pickle

with open("/content/code_graph.pkl", "wb") as f:
    pickle.dump(G, f)

print("Graph saved successfully!")

Graph saved successfully!


In [16]:
from collections import Counter

relationship_counts = Counter(
    data["relation"]
    for _, _, data in G.edges(data=True)
)

for relation, count in relationship_counts.items():
    print(relation, ":", count)

IMPORTS : 203
DEFINED_IN : 288
CALLS : 472


In [17]:
import networkx as nx
import numpy as np
import random
from collections import Counter

# Create a graph containing only CALLS relationships
CALLS_G = nx.DiGraph()

# Keep all function nodes as possible candidates
for node, data in G.nodes(data=True):
    if data.get("type") == "Function":
        CALLS_G.add_node(node, **data)

# Add only CALLS edges
for u, v, data in G.edges(data=True):
    if data.get("relation") == "CALLS":
        CALLS_G.add_edge(u, v)

print("CALLS graph created")
print("Nodes:", CALLS_G.number_of_nodes())
print("Edges:", CALLS_G.number_of_edges())

print("Node types:", Counter(
    data.get("type") for _, data in CALLS_G.nodes(data=True)
))

CALLS graph created
Nodes: 241
Edges: 472
Node types: Counter({'Function': 241})


In [18]:
# Common settings
DAMPING = 0.85
TOP_K = 5


def get_candidates(graph):
    return [
        node for node, data in graph.nodes(data=True)
        if data.get("type") == "Function"
    ]


def validate_seeds(graph, seeds):
    valid = [s for s in seeds if s in graph]

    if not valid:
        raise ValueError(
            "None of the supplied seed nodes exists in the CALLS graph."
        )

    return valid


# -------------------------------------------------
# 1. PERSONALIZED PAGERANK (PPR)
# -------------------------------------------------

def personalized_pagerank(graph, seeds, top_k=TOP_K):

    seeds = validate_seeds(graph, seeds)

    personalization = {node: 0.0 for node in graph.nodes()}

    for seed in seeds:
        personalization[seed] = 1 / len(seeds)

    scores = nx.pagerank(
        graph,
        alpha=DAMPING,
        personalization=personalization,
        max_iter=200,
        tol=1e-8
    )

    candidates = get_candidates(graph)

    ranked = sorted(
        candidates,
        key=lambda node: scores.get(node, 0),
        reverse=True
    )

    return ranked[:top_k], scores


# -------------------------------------------------
# 2. RANDOM WALK WITH RESTART (RWR)
# -------------------------------------------------

def random_walk_restart(
    graph, seeds, top_k=TOP_K,
    walks=5000, max_steps=30,
    restart_probability=0.15, random_seed=42
):

    seeds = validate_seeds(graph, seeds)
    rng = random.Random(random_seed)

    visits = Counter()

    for _ in range(walks):

        current = rng.choice(seeds)

        for _ in range(max_steps):

            visits[current] += 1

            # Restart from the symptom seed
            if rng.random() < restart_probability:
                current = rng.choice(seeds)
                continue

            neighbors = list(graph.successors(current))

            if neighbors:
                current = rng.choice(neighbors)
            else:
                current = rng.choice(seeds)

    total_visits = sum(visits.values())

    scores = {
        node: visits[node] / total_visits
        for node in graph.nodes()
    }

    candidates = get_candidates(graph)

    ranked = sorted(
        candidates,
        key=lambda node: scores.get(node, 0),
        reverse=True
    )

    return ranked[:top_k], scores


# -------------------------------------------------
# 3. PERSONALIZED HITS
# -------------------------------------------------

def personalized_hits(
    graph, seeds, top_k=TOP_K, k_hops=2
):

    seeds = validate_seeds(graph, seeds)

    # Find nodes within k hops of the symptom seeds
    undirected = graph.to_undirected()

    local_nodes = set(seeds)

    for seed in seeds:
        distances = nx.single_source_shortest_path_length(
            undirected, seed, cutoff=k_hops
        )
        local_nodes.update(distances.keys())

    # Construct the local directed ego-subgraph
    local_graph = graph.subgraph(local_nodes).copy()

    if local_graph.number_of_edges() == 0:
        raise ValueError(
            "The local HITS subgraph has no edges. "
            "Try increasing k_hops or checking the seed."
        )

    hubs, authorities = nx.hits(
        local_graph,
        max_iter=1000,
        tol=1e-8,
        normalized=True
    )

    candidates = [
        node for node in local_graph.nodes()
        if local_graph.nodes[node].get("type") == "Function"
    ]

    ranked = sorted(
        candidates,
        key=lambda node: authorities.get(node, 0),
        reverse=True
    )

    return ranked[:top_k], authorities


# -------------------------------------------------
# 4. HYBRID PPR + HITS USING RRF
# -------------------------------------------------

def hybrid_ppr_hits(
    graph, seeds, top_k=TOP_K,
    rrf_k=60, k_hops=2
):

    ppr_ranked, ppr_scores = personalized_pagerank(
        graph, seeds, top_k=len(get_candidates(graph))
    )

    hits_ranked, hits_scores = personalized_hits(
        graph, seeds,
        top_k=len(get_candidates(graph)),
        k_hops=k_hops
    )

    # Reciprocal Rank Fusion
    rrf_scores = Counter()

    for rank, node in enumerate(ppr_ranked, start=1):
        rrf_scores[node] += 1 / (rrf_k + rank)

    for rank, node in enumerate(hits_ranked, start=1):
        rrf_scores[node] += 1 / (rrf_k + rank)

    ranked = sorted(
        rrf_scores,
        key=lambda node: rrf_scores[node],
        reverse=True
    )

    return ranked[:top_k], rrf_scores


print("All four algorithms are ready!")

All four algorithms are ready!


In [19]:
# Replace this with an exact function node printed in Step 11
SEEDS = [
    "src/requests/api.py::get"
]

# Run PPR
ppr_results, ppr_scores = personalized_pagerank(
    CALLS_G, SEEDS
)

# Run RWR
rwr_results, rwr_scores = random_walk_restart(
    CALLS_G, SEEDS
)

# Run Personalized HITS
hits_results, hits_scores = personalized_hits(
    CALLS_G, SEEDS, k_hops=2
)

# Run Hybrid
hybrid_results, hybrid_scores = hybrid_ppr_hits(
    CALLS_G, SEEDS, rrf_k=60, k_hops=2
)

print("PPR:", ppr_results)
print("\nRWR:", rwr_results)
print("\nPersonalized HITS:", hits_results)
print("\nHybrid:", hybrid_results)

PPR: ['src/requests/api.py::get', 'src/requests/api.py::request', 'src/requests/sessions.py::Session.request', 'src/requests/cookies.py::RequestsCookieJar.set_cookie', 'src/requests/cookies.py::RequestsCookieJar.update']

RWR: ['src/requests/api.py::get', 'src/requests/sessions.py::Session.request', 'src/requests/api.py::request', 'src/requests/cookies.py::RequestsCookieJar.set_cookie', 'src/requests/cookies.py::RequestsCookieJar.update']

Personalized HITS: ['src/requests/cookies.py::RequestsCookieJar.get', 'src/requests/structures.py::LookupDict.get', 'src/requests/api.py::get', 'src/requests/sessions.py::Session.get', 'src/requests/adapters.py::BaseAdapter.send']

Hybrid: ['src/requests/api.py::get', 'src/requests/adapters.py::BaseAdapter.send', 'src/requests/cookies.py::RequestsCookieJar.get', 'src/requests/sessions.py::SessionRedirectMixin.send', 'src/requests/adapters.py::HTTPAdapter.send']


In [20]:
# Replace this with an exact function node printed in Step 11
SEEDS = [
    "src/requests/api.py::get"
]

# Run PPR
ppr_results, ppr_scores = personalized_pagerank(
    CALLS_G, SEEDS
)

# Run RWR
rwr_results, rwr_scores = random_walk_restart(
    CALLS_G, SEEDS
)

# Run Personalized HITS
hits_results, hits_scores = personalized_hits(
    CALLS_G, SEEDS, k_hops=2
)

# Run Hybrid
hybrid_results, hybrid_scores = hybrid_ppr_hits(
    CALLS_G, SEEDS, rrf_k=60, k_hops=2
)

print("PPR:", ppr_results)
print("\nRWR:", rwr_results)
print("\nPersonalized HITS:", hits_results)
print("\nHybrid:", hybrid_results)

PPR: ['src/requests/api.py::get', 'src/requests/api.py::request', 'src/requests/sessions.py::Session.request', 'src/requests/cookies.py::RequestsCookieJar.set_cookie', 'src/requests/cookies.py::RequestsCookieJar.update']

RWR: ['src/requests/api.py::get', 'src/requests/sessions.py::Session.request', 'src/requests/api.py::request', 'src/requests/cookies.py::RequestsCookieJar.set_cookie', 'src/requests/cookies.py::RequestsCookieJar.update']

Personalized HITS: ['src/requests/cookies.py::RequestsCookieJar.get', 'src/requests/structures.py::LookupDict.get', 'src/requests/api.py::get', 'src/requests/sessions.py::Session.get', 'src/requests/adapters.py::BaseAdapter.send']

Hybrid: ['src/requests/api.py::get', 'src/requests/adapters.py::BaseAdapter.send', 'src/requests/cookies.py::RequestsCookieJar.get', 'src/requests/sessions.py::SessionRedirectMixin.send', 'src/requests/adapters.py::HTTPAdapter.send']


In [21]:
results = {
    "PPR": ppr_results,
    "RWR": rwr_results,
    "HITS": hits_results,
    "Hybrid": hybrid_results
}

print("TOP-5 OVERLAP")

for a in results:
    for b in results:
        if a < b:
            overlap = set(results[a]) & set(results[b])

            print(
                f"{a} vs {b}: "
                f"{len(overlap)}/5 common functions"
            )
            print("Common:", overlap)
            print()

TOP-5 OVERLAP
PPR vs RWR: 5/5 common functions
Common: {'src/requests/cookies.py::RequestsCookieJar.update', 'src/requests/api.py::get', 'src/requests/cookies.py::RequestsCookieJar.set_cookie', 'src/requests/sessions.py::Session.request', 'src/requests/api.py::request'}

HITS vs PPR: 1/5 common functions
Common: {'src/requests/api.py::get'}

HITS vs RWR: 1/5 common functions
Common: {'src/requests/api.py::get'}

HITS vs Hybrid: 3/5 common functions
Common: {'src/requests/adapters.py::BaseAdapter.send', 'src/requests/cookies.py::RequestsCookieJar.get', 'src/requests/api.py::get'}

Hybrid vs PPR: 1/5 common functions
Common: {'src/requests/api.py::get'}

Hybrid vs RWR: 1/5 common functions
Common: {'src/requests/api.py::get'}



In [22]:
SEEDS = [
    "src/requests/api.py::get"
]

TOP_K = 5

In [23]:
# Get rankings beyond the top 5 so we can remove seeds first
all_candidates = len(CALLS_G.nodes())

ppr_ranked, _ = personalized_pagerank(
    CALLS_G, SEEDS, top_k=all_candidates
)

rwr_ranked, _ = random_walk_restart(
    CALLS_G, SEEDS, top_k=all_candidates
)

hits_ranked, _ = personalized_hits(
    CALLS_G, SEEDS, top_k=all_candidates
)

hybrid_ranked, _ = hybrid_ppr_hits(
    CALLS_G, SEEDS, top_k=all_candidates
)

# Remove symptom seed nodes and retain top 5
def remove_seeds(ranking, seeds, top_k=5):
    return [
        node for node in ranking if node not in seeds
    ][:top_k]

ppr_top5 = remove_seeds(ppr_ranked, SEEDS, TOP_K)
rwr_top5 = remove_seeds(rwr_ranked, SEEDS, TOP_K)
hits_top5 = remove_seeds(hits_ranked, SEEDS, TOP_K)
hybrid_top5 = remove_seeds(hybrid_ranked, SEEDS, TOP_K)

print("PPR:", ppr_top5)
print("\nRWR:", rwr_top5)
print("\nHITS:", hits_top5)
print("\nHybrid:", hybrid_top5)

PPR: ['src/requests/api.py::request', 'src/requests/sessions.py::Session.request', 'src/requests/cookies.py::RequestsCookieJar.set_cookie', 'src/requests/cookies.py::RequestsCookieJar.update', 'src/requests/adapters.py::BaseAdapter.send']

RWR: ['src/requests/sessions.py::Session.request', 'src/requests/api.py::request', 'src/requests/cookies.py::RequestsCookieJar.set_cookie', 'src/requests/cookies.py::RequestsCookieJar.update', 'src/requests/adapters.py::HTTPAdapter.send']

HITS: ['src/requests/cookies.py::RequestsCookieJar.get', 'src/requests/structures.py::LookupDict.get', 'src/requests/sessions.py::Session.get', 'src/requests/adapters.py::BaseAdapter.send', 'src/requests/sessions.py::SessionRedirectMixin.send']

Hybrid: ['src/requests/adapters.py::BaseAdapter.send', 'src/requests/cookies.py::RequestsCookieJar.get', 'src/requests/sessions.py::SessionRedirectMixin.send', 'src/requests/adapters.py::HTTPAdapter.send', 'src/requests/structures.py::LookupDict.get']


In [24]:
rankings = {
    "PPR": set(ppr_top5),
    "RWR": set(rwr_top5),
    "HITS": set(hits_top5),
    "Hybrid": set(hybrid_top5)
}

pairs = [
    ("PPR", "RWR"),
    ("HITS", "PPR"),
    ("HITS", "RWR"),
    ("HITS", "Hybrid"),
    ("Hybrid", "PPR"),
    ("Hybrid", "RWR")
]

for a, b in pairs:
    common = rankings[a] & rankings[b]
    print(f"{a} vs {b}: {len(common)}/5 common functions")
    print("Common:", common)
    print()

PPR vs RWR: 4/5 common functions
Common: {'src/requests/sessions.py::Session.request', 'src/requests/cookies.py::RequestsCookieJar.update', 'src/requests/api.py::request', 'src/requests/cookies.py::RequestsCookieJar.set_cookie'}

HITS vs PPR: 1/5 common functions
Common: {'src/requests/adapters.py::BaseAdapter.send'}

HITS vs RWR: 0/5 common functions
Common: set()

HITS vs Hybrid: 4/5 common functions
Common: {'src/requests/structures.py::LookupDict.get', 'src/requests/sessions.py::SessionRedirectMixin.send', 'src/requests/cookies.py::RequestsCookieJar.get', 'src/requests/adapters.py::BaseAdapter.send'}

Hybrid vs PPR: 1/5 common functions
Common: {'src/requests/adapters.py::BaseAdapter.send'}

Hybrid vs RWR: 1/5 common functions
Common: {'src/requests/adapters.py::HTTPAdapter.send'}

